In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

repo_root = Path.cwd().parent
sys.path.append(str(repo_root))

# import local libraries
import src.utils.config as config
import proc.process_behavior_signals as pbs
# import proc.organize as org
import src.utils.data_io as dio
import src.utils.pdata_io as pdio
import numpy as np

# print(config.PROJECT_ROOT)
# print(config.RAW_BASE)
# print(config.PROC_BASE)

import sys
from pathlib import Path

CODE_BASE = Path(config.CODE_BASE).expanduser()
sys.path.append(str(CODE_BASE))

print("Added to path:", CODE_BASE)

# make the data dictionary which contains list of all animals and sessions and
# file names of videos h264 and mat files

data_root = config.RAW_BASE # this variable now contains the path to the raw data directory, which is defined in the config file. It is used to build the classical conditioning data dictionary using the dio.build_classical_conditioning_dict function. The resulting cc_data variable will contain the structured information about the animals, sessions, and file names for the classical conditioning experiments.
cc_data = dio.build_classical_conditioning_dict(data_root)

pdata_root = config.PROC_BASE # this variable now contains the path to the processed data directory, which is defined in the config file. It is used to build the processed data dictionary using the pdio.build_processed_data_dict function. The resulting pdata variable will contain the structured information about the animals, sessions, and file names for the processed data of the classical conditioning experiments.
print(data_root)
print(pdata_root)

In [18]:
import pandas as pd


def summarize_cc_data_phases(cc_data):
    """
    Summarize how many sessions each animal has in each phase.

    Expected cc_data structure:
        cc_data[animal][date]["phase"]

    Returns
    -------
    summary_df : pandas DataFrame
        One row per animal with number of sessions per phase.

    long_df : pandas DataFrame
        One row per animal/date/session.
    """

    long_rows = []

    for animal, sessions in cc_data.items():

        for date, info in sessions.items():

            phase = info.get("phase", "unknown")

            long_rows.append({
                "animal": animal,
                "date": date,
                "phase": phase,
                "has_face": info.get("face") is not None,
                "has_pupi": info.get("pupi") is not None,
                "has_video": info.get("video") is not None,
                "has_recording": info.get("recording") is not None,
                "path": info.get("path"),
            })

    long_df = pd.DataFrame(long_rows)

    if long_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    summary_df = (
        long_df
        .groupby(["animal", "phase"])
        .size()
        .reset_index(name="n_sessions")
        .pivot(index="animal", columns="phase", values="n_sessions")
        .fillna(0)
        .astype(int)
        .reset_index()
    )

    return summary_df, long_df

In [19]:
summary_df, long_df = summarize_cc_data_phases(cc_data)

summary_df

phase,animal,air_training,habituation,tone_air_training,unknown
0,NML_04,15,16,10,11
1,NML_05,15,16,10,11
2,NML_06,15,16,10,11
3,NML_07,18,19,10,11
4,NML_08,18,19,10,11
